# Lab 18: How to choose an agent design

**Notebook 2 of 2: agent designs at the Chinook support desk**

## Goal

Take the designs from the book club to the support desk of the Chinook music shop, where a rep asks about customers
and the facts are in the shop's database. The rep asks for five customers' totals and who spent the most. One agent
with the desk's lookups answers it; so do five sub-agents, one per customer, whose five lines go to an agent that
writes the answer. We print the whole input of that answering agent: five lines, against everything one agent read.
Then requests that arrive from a form with a label are answered by code with no model call, and a wrong label shows
what that costs. The notebook ends with the four reasons to add a second agent, each beside what this lab measured.

## What you will learn

- How to print everything the agent that writes the answer read, and compare it with one agent's on the same request.
- How to split independent lookups across parallel sub-agents, and say what the split costs in requests and tokens.
- How to answer a form's labelled requests in code, send the rest to an agent, and see what a wrong label does.
- How to decide whether a second agent is worth it with four reasons: different permissions, a smaller context,
  parallel work, and another team or model.

## From the book club to the desk

Notebook 1 used these ideas, and we keep them here:

- The **answering request** is the model request that writes the answer, the last of a run. Its input tokens measure
  everything the model read just before it answered.
- A **sub-agent** is an agent started for one part of a job; it does its own reading and returns only its result.
  Our code can start several at the same time with `asyncio.gather` and pass their results to an agent that answers.
- A **hybrid** answers in code the requests whose label says what to do, and sends the rest to an agent.
- A run's request and token counts come from two objects: `run.context_wrapper.usage` adds up every model request of the run (`.requests` counts them, `.input_tokens` sums what they read), and `run.raw_responses[-1].usage` holds the counts of the answering request alone.

Here a wrong answer costs more: the facts come from a shop's database, and a rep will repeat them to a customer. This
notebook restates everything it uses, so it runs on its own.

## Setup

The first cell is the same in every lab: it finds the repository folder, reads the model and prices from
`models.env`, and takes the key from `.env` without printing it.

In [1]:
from pathlib import Path
import asyncio
import os
import sqlite3
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


Two settings go to every agent here: a client that waits at most 60 seconds for a reply, so a stuck request fails
quickly, and `SETTINGS`, which passes the reasoning effort from `models.env` to every model request.

In [2]:
from openai import AsyncOpenAI
from openai.types.shared import Reasoning
from agents import Agent, ModelSettings, Runner, function_tool, set_default_openai_client

set_default_openai_client(AsyncOpenAI(timeout=60, max_retries=2))
SETTINGS = ModelSettings(reasoning=Reasoning(effort=EFFORT))

---

## Part A: Look up five customers' totals by hand

The shop's data is `data/chinook.db`, a SQLite file with the customers, their invoices and the support reps who look
after them. The desk has three lookups, one small function each:

- `find_customer(name)` finds a customer by full name and returns their `customer_id`.
- `get_invoices(customer_id)` counts that customer's invoices and adds up their total.
- `get_support_rep(customer_id)` names the rep who looks after them.

`mode=ro` opens the database read-only, so no tool can change the shop's data. Each lookup goes through `query`,
which opens a connection of its own and closes it when the query is done (`closing(...)` closes it at the end of the
`with` block). The type hints and docstrings are what `function_tool` turns into each tool's description.

**Why a connection per lookup?** When the model asks for several lookups in one reply, the SDK runs them at the same
time, each in its own thread, and one SQLite connection used by several threads at once is not safe. In a test of 200
rounds of five lookups at the same time on one shared connection, about one lookup in ten came back empty, with
another customer's row, or with `InterfaceError: bad parameter or other API misuse`, and an agent would read those
results as facts. With a connection per lookup, all 1,000 were right. The extra notebook runs the same test.

In [3]:
from contextlib import closing

DB_URI = f"file:{ROOT / 'data' / 'chinook.db'}?mode=ro"


def query(sql, args):
    """Run one read-only query on a connection of its own, and return every row."""
    with closing(sqlite3.connect(DB_URI, uri=True)) as conn:
        return conn.execute(sql, args).fetchall()


def find_customer(name: str) -> list[dict]:
    """Find a shop customer by full name. Returns a list of matches with their customer_id.

    Args:
        name: First and last name, for example 'Ana Silva'.
    """
    rows = query("SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
                 "WHERE FirstName || ' ' || LastName = ?", (name,))
    return [{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows]


def get_invoices(customer_id: int) -> dict:
    """Count a customer's invoices and add up their total in US dollars."""
    count, total = query("SELECT COUNT(*), ROUND(SUM(Total), 2) FROM invoices WHERE CustomerId = ?", (customer_id,))[0]
    return {"customer_id": customer_id, "invoice_count": count, "total": total}


def get_support_rep(customer_id: int) -> dict:
    """Name the support rep who looks after a customer."""
    rows = query("SELECT e.FirstName || ' ' || e.LastName FROM customers c "
                 "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,))
    return {"customer_id": customer_id, "support_rep": rows[0][0] if rows else None}

The rep's request for this notebook names five customers. We look up their totals by hand first, so we know the
right answer before any model is involved.

In [4]:
FIVE = ["Helena Holý", "Puja Srivastava", "Richard Cunningham", "Luis Rojas", "Manoj Pareek"]

for name in FIVE:
    customer_id = find_customer(name)[0]["customer_id"]
    print(f"{name:20} customer {customer_id:2}  total ${get_invoices(customer_id)['total']:.2f}")

Helena Holý          customer  6  total $49.62
Puja Srivastava      customer 59  total $36.64
Richard Cunningham   customer 26  total $47.62
Luis Rojas           customer 57  total $46.62
Manoj Pareek         customer 58  total $38.62


These are the facts every answer in this notebook is checked against:

| Customer | Total spent |
|---|---|
| Helena Holý | \$49.62 (the most) |
| Puja Srivastava | \$36.64 |
| Richard Cunningham | \$47.62 |
| Luis Rojas | \$46.62 |
| Manoj Pareek | \$38.62 |

Each total needs two lookups (find the customer, then read the invoices), and no customer's lookups depend on another
customer's: the five parts of the request are independent.

In [5]:
TOTALS = ("What has each of these five customers spent in total: Helena Holý, Puja Srivastava, Richard Cunningham, "
          "Luis Rojas and Manoj Pareek? Who spent the most?")

**Checkpoint A.** Five totals from \$36.64 to \$49.62, with Helena Holý the highest; ten lookups in all, two per
customer.

---

## Part B: Measure what one agent reads before it answers

The baseline is the desk agent: one agent with the three lookups, told to look up every fact.

In [6]:
desk = Agent(
    name="Desk",
    instructions=("You help the support reps of a music shop. Look up every fact with the tools; "
                  "never answer from memory. Answer in one or two sentences."),
    model=MODEL,
    tools=[function_tool(find_customer), function_tool(get_invoices), function_tool(get_support_rep)],
    model_settings=SETTINGS,
)

result = await Runner.run(desk, TOTALS)
print(result.final_output)

Helena Holý spent $49.62; Puja Srivastava $36.64; Richard Cunningham $47.62; Luis Rojas $46.62; and Manoj Pareek $38.62. **Helena Holý spent the most**, at $49.62.


All five totals, matching Part A's table, and Helena Holý named as the customer who spent the most.

For each model request, its input tokens and what it asked for:

In [7]:
for number, response in enumerate(result.raw_responses, start=1):
    asked = [getattr(part, "name", part.type) for part in response.output]
    print(f"request {number}: {response.usage.input_tokens:4} input tokens -> {asked}")

request 1:  222 input tokens -> ['find_customer', 'find_customer', 'find_customer', 'find_customer', 'find_customer']
request 2:  494 input tokens -> ['get_invoices', 'get_invoices', 'get_invoices', 'get_invoices', 'get_invoices']
request 3:  756 input tokens -> ['message']


Three requests, each reading more than the one before:

1. 222 tokens: the instructions and the request. The model asked for `find_customer` five times in one reply, because the five names do not depend on each other.
2. 494 tokens: now with the five ids. It asked for `get_invoices` five times in one reply.
3. 756 tokens: the answering request, carrying all ten calls and their results. It wrote the answer.

What the answering request read, item by item (`to_input_list()` without its last item, the answer):

In [8]:
for item in result.to_input_list()[:-1]:
    kind = item.get("type", "message")
    if kind == "function_call":
        print("tool call   |", item["name"], item["arguments"])
    elif kind == "function_call_output":
        print("tool result |", item["output"])
    else:
        print("question    |", item["content"][:70], "...")

question    | What has each of these five customers spent in total: Helena Holý, Puj ...
tool call   | find_customer {"name":"Helena Holý"}
tool call   | find_customer {"name":"Puja Srivastava"}
tool call   | find_customer {"name":"Richard Cunningham"}
tool call   | find_customer {"name":"Luis Rojas"}
tool call   | find_customer {"name":"Manoj Pareek"}
tool result | [{'customer_id': 6, 'name': 'Helena Holý', 'country': 'Czech Republic'}]
tool result | [{'customer_id': 59, 'name': 'Puja Srivastava', 'country': 'India'}]
tool result | [{'customer_id': 26, 'name': 'Richard Cunningham', 'country': 'USA'}]
tool result | [{'customer_id': 57, 'name': 'Luis Rojas', 'country': 'Chile'}]
tool result | [{'customer_id': 58, 'name': 'Manoj Pareek', 'country': 'India'}]
tool call   | get_invoices {"customer_id":6}
tool call   | get_invoices {"customer_id":59}
tool call   | get_invoices {"customer_id":26}
tool call   | get_invoices {"customer_id":57}
tool call   | get_invoices {"customer_id":58}
tool

The request, ten tool calls and ten results: every customer's id and country, then every invoice count and total. To write five totals and one name, the answering request read every step that found them.

**Checkpoint B.** The desk agent gives all five totals and names Helena Holý. Its answering request read the question,
every lookup call and every result (756 input tokens in our run), in 3 requests.

---

## Part C: Split the lookups across five sub-agents and print what the answering agent reads

Now the design from notebook 1: one sub-agent per customer, all running at the same time, each returning one line,
and an agent with no tools that writes the answer from the five lines.

### One worker on one customer

The **worker** has only the two lookups a total needs and is told to reply with one line.

In [9]:
worker = Agent(
    name="Customer worker",
    instructions=("You look up one customer's total spend. Look up every fact with your tools; never answer from "
                  "memory. Reply with one line: the name and the total in dollars."),
    model=MODEL,
    tools=[function_tool(find_customer), function_tool(get_invoices)],
    model_settings=SETTINGS,
)

one_worker = await Runner.run(worker, "What has Helena Holý spent in total?")
print(one_worker.final_output)
print("requests:", one_worker.context_wrapper.usage.requests)

Helena Holý — $49.62
requests: 3


One line, *Helena Holý — \$49.62*, from 3 requests: find the customer, read the invoices, write the line. The worker's lookups stay inside its own run.

### Five workers at the same time

`asyncio.gather` starts the five runs together and returns their results in the order of `FIVE`, so `zip(FIVE,
worker_runs)` pairs each name with its own result. `trace(...)` records the five runs under one name on the Traces
page.

In [10]:
from agents import trace

with trace("Five customers' totals"):
    worker_runs = await asyncio.gather(
        *[Runner.run(worker, f"What has {name} spent in total?") for name in FIVE])

for name, run in zip(FIVE, worker_runs):
    print(f"{name:20} -> {run.final_output}   ({run.context_wrapper.usage.requests} requests)")

Helena Holý          -> Helena Holý — $49.62   (3 requests)
Puja Srivastava      -> Puja Srivastava — $36.64   (3 requests)
Richard Cunningham   -> Richard Cunningham — $47.62   (3 requests)
Luis Rojas           -> Luis Rojas — $46.62   (3 requests)
Manoj Pareek         -> Manoj Pareek — $38.62   (3 requests)


Five lines with the five right totals, each from 3 requests.

### The main agent reads five lines

The **main** agent has no tools. Its input is the rep's request and the five lines; we print it in full, because it
is everything the agent will read before it answers.

In [11]:
main = Agent(
    name="Main",
    instructions="You help the support reps of a music shop. Answer from the lines you are given; do not add facts.",
    model=MODEL,
    model_settings=SETTINGS,
)

main_input = TOTALS + "\n\nWhat the workers found:\n" + "\n".join(run.final_output for run in worker_runs)
print(main_input)

What has each of these five customers spent in total: Helena Holý, Puja Srivastava, Richard Cunningham, Luis Rojas and Manoj Pareek? Who spent the most?

What the workers found:
Helena Holý — $49.62
Puja Srivastava — $36.64
Richard Cunningham — $47.62
Luis Rojas — $46.62
Manoj Pareek — $38.62


In [12]:
joined = await Runner.run(main, main_input)

print(joined.final_output)
print()
print("answering request:", joined.raw_responses[-1].usage.input_tokens, "input tokens")

- Helena Holý: $49.62
- Puja Srivastava: $36.64
- Richard Cunningham: $47.62
- Luis Rojas: $46.62
- Manoj Pareek: $38.62

Helena Holý spent the most, at **$49.62**.

answering request: 125 input tokens


The same five totals and the same top spender as one agent. The main agent's answering request read 125 tokens: the instructions, the request and five lines. One agent's answering request read 756.

### What the split cost

In [13]:
split_runs = [*worker_runs, joined]

print("one agent:            requests", result.context_wrapper.usage.requests,
      "| input tokens in total", result.context_wrapper.usage.input_tokens,
      "| answering request", result.raw_responses[-1].usage.input_tokens)
print("five workers + main:  requests", sum(run.context_wrapper.usage.requests for run in split_runs),
      "| input tokens in total", sum(run.context_wrapper.usage.input_tokens for run in split_runs),
      "| answering request", joined.raw_responses[-1].usage.input_tokens)

one agent:            requests 3 | input tokens in total 1472 | answering request 756
five workers + main:  requests 16 | input tokens in total 3364 | answering request 125


- **Answering request:** 125 tokens against 756, about a sixth, as in notebook 1.
- **Requests:** 16 against 3: five workers of 3 requests each, and the main agent's one.
- **Input tokens in total:** 3,364 against 1,472 (one agent's three requests, 222 + 494 + 756), more than twice as many. Each worker sends its own instructions and request on each of its three requests.

Read the sizes as well as the ratio. Each lookup here returns one short line of data, so one agent's answering request was only 756 tokens, and saving 631 of them is not worth 13 more requests. The split pays when each part reads much more than it returns: the blurbs in notebook 1, or here a customer's long ticket history.

**Checkpoint C.** One worker: one line, 3 requests. Five workers: five lines with the right totals. The main agent's
whole input is the request and five lines (125 input tokens in our run, against 756 for one agent),
and its answer names Helena Holý. The split took 16 requests against one agent's 3.

---

## Part D: Answer the form's labelled requests in code and send the rest to the agent

Many requests reach a desk through a form: the rep picks the kind of request from a list and types a name. The kind
is a label that tells our code the steps, so it can run the lookups with no model: no tokens, no waiting, the same
path every time. The agent is kept for what the form does not cover. This is the hybrid from notebook 1, on a form
instead of a chat command.

### The form's requests

Each request is a dictionary. `kind` is the label the rep picked; the other fields are what the form asked for. The
last request is free text, so its kind is `open`.

In [14]:
FORM = [
    {"kind": "support_rep", "customer": "Helena Holý"},
    {"kind": "invoices", "customer": "Puja Srivastava"},
    {"kind": "totals", "customers": FIVE},
    {"kind": "open", "text": "Has Helena Holý spent more than Puja Srivastava? By how much?"},
]
for request in FORM:
    print(request)

{'kind': 'support_rep', 'customer': 'Helena Holý'}
{'kind': 'invoices', 'customer': 'Puja Srivastava'}
{'kind': 'totals', 'customers': ['Helena Holý', 'Puja Srivastava', 'Richard Cunningham', 'Luis Rojas', 'Manoj Pareek']}
{'kind': 'open', 'text': 'Has Helena Holý spent more than Puja Srivastava? By how much?'}


### A handler for each label

Each handler is the Part A lookups in a fixed order and a sentence with blanks to fill. `max(totals, key=totals.get)`
returns the dictionary key whose value is largest, here the name of the customer who spent the most.

In [15]:
def customer_id_of(name):
    return find_customer(name)[0]["customer_id"]


def answer_support_rep(request):
    rep = get_support_rep(customer_id_of(request["customer"]))["support_rep"]
    return f"{request['customer']}'s support rep is {rep}."


def answer_invoices(request):
    invoices = get_invoices(customer_id_of(request["customer"]))
    return f"{request['customer']} has {invoices['invoice_count']} invoices adding up to ${invoices['total']:.2f}."


def answer_totals(request):
    totals = {name: get_invoices(customer_id_of(name))["total"] for name in request["customers"]}
    lines = [f"{name}: ${total:.2f}" for name, total in totals.items()]
    return "\n".join(lines) + f"\n{max(totals, key=totals.get)} spent the most."


HANDLERS = {"support_rep": answer_support_rep, "invoices": answer_invoices, "totals": answer_totals}

These handlers work only when the name on the form is in the database. For a name with no match, `find_customer` returns an
empty list and `[0]` raises `IndexError`; a real form would let the rep pick the customer from a list, or the handler
would reply that there is no such customer.

### The router

`route` looks the label up in `HANDLERS`. `HANDLERS.get(kind)` returns the handler, or `None` for a label it does not
have, which is what sends `open` to the desk agent of Part B. It returns the reply and the number of model requests.

In [16]:
async def route(request):
    handler = HANDLERS.get(request["kind"])
    if handler:                                    # a labelled request: our code answers
        return handler(request), 0
    run = await Runner.run(desk, request["text"])  # anything else goes to the agent
    return run.final_output, run.context_wrapper.usage.requests


for request in FORM:
    reply, requests = await route(request)
    print(f"[{request['kind']}] model requests: {requests}\n{reply}\n")

[support_rep] model requests: 0
Helena Holý's support rep is Steve Johnson.

[invoices] model requests: 0
Puja Srivastava has 6 invoices adding up to $36.64.

[totals] model requests: 0
Helena Holý: $49.62
Puja Srivastava: $36.64
Richard Cunningham: $47.62
Luis Rojas: $46.62
Manoj Pareek: $38.62
Helena Holý spent the most.



[open] model requests: 3
Yes. Helena Holý spent **$12.98 more** than Puja Srivastava ($49.62 vs. $36.64).



Our code answered the three labelled requests with 0 model requests, every fact right: Helena's rep Steve Johnson, Puja's 6 invoices and \$36.64, and the five totals with Helena on top. The open request has no handler, so it went to the desk agent, which answered \$12.98 more in 3 requests.

### What happens when the label is wrong

The code goes by the label alone and never reads the text the rep typed. Here is a request whose label does not match
what the rep typed: the rep picked `support_rep` from the list, then typed a question about money in a free-text box.

In [17]:
mislabelled = {"kind": "support_rep", "customer": "Helena Holý", "text": "What has Helena Holý spent in total?"}

reply, requests = await route(mislabelled)
print(reply, "| model requests:", requests)

Helena Holý's support rep is Steve Johnson. | model requests: 0


A correct sentence that answers the wrong question, with nothing to flag it: Helena's rep, when the rep asked what
she spent. The hybrid is fast and costs nothing because the label decides, and it is right only as often as the label
is.

**In production:** the label comes from a form with a fixed list, or from one cheap model call that picks a label
from that list (checked like any other model output). A request that fits none of the labels on the list is marked `open` and goes to the agent.

**Checkpoint D.** The three labelled requests answered with 0 model requests and the right facts; the open request
sent to the desk agent (\$12.98 more, in 3 requests in our run); the mislabelled request answered with Helena's rep (Steve Johnson) instead of
her total, with 0 requests.

---

## Part E: Decide when a second agent is worth its cost

On every request in this lab, one agent gave the right answer, and a second agent did not make any answer more right.
A second agent is worth its extra requests and tokens for one of four reasons:

| Reason | What it means at the desk | What this lab shows |
|---|---|---|
| **Different permissions** | A refund agent may write; the lookup agents may only read. Splitting keeps the write tool away from every other agent | Not run here, since every tool reads. Part C's worker gets only the two lookups it needs, the same way a refund agent would be the only agent given the write tool |
| **A smaller context** | A sub-agent reads a lot and returns one line, so the agent that writes the answer reads little | Part C: the main agent read 125 tokens against one agent's 756; notebook 1: five lines against five whole blurbs |
| **Parallel work** | Independent parts run at the same time | Timed in notebook 1, Part C: five readers took 3.0 seconds against 2.5 for one. One agent can also ask for several lookups in one reply, as Part B's desk agent did, so time alone rarely pays for the split |
| **Another team or model** | Another team owns its agent (for example, the shipping team's agent), or a cheap model handles the simple parts and a strong one answers | Not run: `models.env` sets one model here. Each sub-agent can take its own `model=` |

Without one of these reasons, one agent with more tools is cheaper, faster to change and easier to test. With one,
pick the design that buys it: parallel sub-agents for independent parts that each read much more than they return;
specialist agents that a main agent calls as tools and gets the answers back from (Lab 07), when each part needs its
own tools or its own owner; a handoff, where one agent passes the conversation to another that writes the reply, when
each request belongs to exactly one team; and plain code for the steps you can write down in advance.

**Checkpoint E.** For each of the four reasons, say which part of this lab showed it, or that it was not run here, and
which design you would pick for it.

---

## In production

- **Another team's agent.** When the second agent runs on another team's server, the two talk over a protocol such as
  A2A (Agent2Agent, an open protocol for agents that different teams run), and each publishes an agent card, a JSON
  document that says what the agent does and where to reach it. The extra notebook,
  `18_extra_measure_the_designs.ipynb`, shows one.
- **Traces hold customer data.** Every run here was also recorded on the Traces page of the OpenAI account that owns
  the key, customer names included. `set_tracing_disabled(True)` switches the upload off.

## What you can do now

- Print everything the agent that writes the answer read, and compare it with one agent's (Parts B and C).
- Split independent lookups across parallel sub-agents, and say what the split costs in requests and tokens and what
  it buys (Part C).
- Route a form's labelled requests to code with no model call, send the rest to an agent, and say what a wrong label
  does (Part D).
- Decide whether a second agent is worth it from the four reasons (Part E).

## Troubleshooting

| What you see | Why | Fix |
|---|---|---|
| `sqlite3.InterfaceError: bad parameter or other API misuse`, or a lookup that returns `[]` or another customer's row for a name that exists | one SQLite connection is shared by tools that the SDK runs at the same time in several threads | open a connection per lookup, as `query` does in Part A |
| `sqlite3.OperationalError: unable to open database file` | `data/chinook.db` is missing from the repository folder | restore the file (clone the repository again, or `git checkout data/chinook.db`) |
| `RuntimeError: AgentRunner.run_sync() cannot be called when an event loop is already running.` | `Runner.run_sync` starts its own event loop, and Jupyter already runs one | in a notebook, write `await Runner.run(...)` |
| `TypeError: An asyncio.Future, a coroutine or an awaitable is required` | the list given to `gather` held finished results, because each `Runner.run(...)` inside it was awaited | build the list without `await`, and await only the `gather` |

## Challenge

1. **Grow the request to ten customers.** Add Ladislav Kovács, Hugh O'Reilly, Frank Ralston, Astrid Gruber and Terhi
   Hämäläinen to the five, ask both designs for every total and who spent the most, and print each design's answering
   request. Expected: both answer Helena Holý (\$49.62); one agent's answering request grows by roughly a hundred
   tokens per customer added, and the main agent's by about 17, one short line each (ours: 1,287 and 208,
   against 756 and 125 for five).
2. **Give the form a new label.** Add a `country` handler (the customer's country is in `find_customer`'s result) and
   send `{"kind": "country", "customer": "Puja Srivastava"}` and `{"kind": "refund", "text": "Can Helena Holý get a
   refund for her last invoice?"}` through `route`. Expected: *India* with 0 model requests; the `refund` label is not
   in `HANDLERS`, so the request goes to the desk agent, which has no refund tool: it looks Helena up and says it
   cannot settle a refund from the records it can read.

The worked solution, run with its outputs, is in `solved/18b_agent_designs_at_the_desk_solution.ipynb`.

## Going further

- **The designs over several runs, and an A2A agent card.** `18_extra_measure_the_designs.ipynb` runs one agent and
  the parallel sub-agents five times each on this request and on notebook 1's, puts every run in one table, and builds
  an A2A agent card for the worker agent.
- **Approval before a write.** Bonus B2 puts a person's approval in front of a refund tool on this desk.

## Resources

- [OpenAI Agents SDK: agent orchestration](https://openai.github.io/openai-agents-python/multi_agent/): orchestration
  by a model and by code, and agents run in parallel with `asyncio.gather`
- [OpenAI Agents SDK: results](https://openai.github.io/openai-agents-python/results/): `raw_responses` and
  `to_input_list()`
- [OpenAI Agents SDK: usage](https://openai.github.io/openai-agents-python/usage/): requests and tokens per run and
  per request
- [OpenAI Agents SDK: tracing](https://openai.github.io/openai-agents-python/tracing/): `trace()` and how to switch the
  upload off
- [Python `asyncio`: coroutines and tasks](https://docs.python.org/3/library/asyncio-task.html): `gather`

Back to [notebook 1](18a_meet_agent_designs.ipynb). Next:
[Lab 19: How to defend an agent from untrusted text](../19_Prompt_Injection/README.md).